In [0]:
# 1. Raw 데이터 읽기
df_raw = spark.read.option("multiline", "true").json("/mnt/raw/sdot/sdot_raw.json")
display(df_raw)

sDoTEnv


In [0]:
from pyspark.sql.functions import explode, col

df_flat = df_raw.select(explode(col("sDoTEnv.row")).alias("r")).select("r.*")

df_flat.printSchema()
display(df_flat.limit(5))

root
 |-- ADMINISTRATIVE_DISTRICT: string (nullable = true)
 |-- AUTONOMOUS_DISTRICT: string (nullable = true)
 |-- AVG_CO: string (nullable = true)
 |-- AVG_EFFE_TEMP: string (nullable = true)
 |-- AVG_H2S: string (nullable = true)
 |-- AVG_HUMI: string (nullable = true)
 |-- AVG_INTE_ILLU: string (nullable = true)
 |-- AVG_NH3: string (nullable = true)
 |-- AVG_NO2: string (nullable = true)
 |-- AVG_NOISE: string (nullable = true)
 |-- AVG_O3: string (nullable = true)
 |-- AVG_SO2: string (nullable = true)
 |-- AVG_TEMP: string (nullable = true)
 |-- AVG_ULTRA_RAYS: string (nullable = true)
 |-- AVG_VIBR_X: string (nullable = true)
 |-- AVG_VIBR_Y: string (nullable = true)
 |-- AVG_VIBR_Z: string (nullable = true)
 |-- AVG_WIND_DIRE: string (nullable = true)
 |-- AVG_WIND_SPEED: string (nullable = true)
 |-- DATA_NO: string (nullable = true)
 |-- DATE: string (nullable = true)
 |-- MAX_CO: string (nullable = true)
 |-- MAX_EFFE_TEMP: string (nullable = true)
 |-- MAX_H2S: string (nul

ADMINISTRATIVE_DISTRICT,AUTONOMOUS_DISTRICT,AVG_CO,AVG_EFFE_TEMP,AVG_H2S,AVG_HUMI,AVG_INTE_ILLU,AVG_NH3,AVG_NO2,AVG_NOISE,AVG_O3,AVG_SO2,AVG_TEMP,AVG_ULTRA_RAYS,AVG_VIBR_X,AVG_VIBR_Y,AVG_VIBR_Z,AVG_WIND_DIRE,AVG_WIND_SPEED,DATA_NO,DATE,MAX_CO,MAX_EFFE_TEMP,MAX_H2S,MAX_HUMI,MAX_INTE_ILLU,MAX_NH3,MAX_NO2,MAX_NOISE,MAX_O3,MAX_SO2,MAX_TEMP,MAX_ULTRA_RAYS,MAX_VIBR_X,MAX_VIBR_Y,MAX_VIBR_Z,MAX_WIND_DIRE,MAX_WIND_SPEED,MIN_CO,MIN_EFFE_TEMP,MIN_H2S,MIN_HUMI,MIN_INTE_ILLU,MIN_NH3,MIN_NO2,MIN_NOISE,MIN_O3,MIN_SO2,MIN_TEMP,MIN_ULTRA_RAYS,MIN_VIBR_X,MIN_VIBR_Y,MIN_VIBR_Z,MIN_WIND_DIRE,MIN_WIND_SPEED,MODELNAME,REGION,SENSING_TIME,SERIAL
Seongsu2ga1(il)-dong,Seongdong-gu,,,,,1815,,,45,,,11.0,0.1,0.05,0.02,1.08,,,1,2026-04-10 11:07:16.0,,,,,2498,,,48,,,11.4,0.1,0.05,0.03,1.08,,,,,,,1283,,,40,,,10.6,0.1,0.05,0.02,1.07,,,SDOT001,residential_area,2026-04-10_11:07:00,V02Q1940556
Majang-dong,Seongdong-gu,,,,87.,0,,,45,,,11.0,0.0,0.02,0.1,1.02,,,1,2026-04-10 11:07:16.0,,,,88.,0,,,48,,,11.2,0.0,0.02,0.1,1.03,,,,,,87.,0,,,41,,,10.8,0.0,0.02,0.1,1.02,,,SDOT001,residential_area,2026-04-10_11:07:00,V02Q1940156
Yongdap-dong,Seongdong-gu,,,,89.,2629,,,48,,,10.8,0.2,0.03,0.07,1.07,,,1,2026-04-10 11:07:16.0,,,,,3357,,,50,,,11.1,0.2,0.03,0.07,1.07,,,,,,89.,2186,,,46,,,10.6,0.1,0.03,0.07,1.06,,,SDOT001,residential_area,2026-04-10_11:07:00,V02Q1940603
Seongsu1ga1(il)-dong,Seongdong-gu,,,,,1312,,,51,,,10.4,0.1,0.02,0.02,1.04,,,1,2026-04-10 11:07:16.0,,,,,1621,,,54,,,10.6,0.1,0.03,0.02,1.04,,,,,,,1096,,,49,,,10.1,0.1,0.02,0.02,1.03,,,SDOT001,residential_area,2026-04-10_11:07:00,V02Q1940502
Geumho4(sa)-dong,Seongdong-gu,,,,,0,,,46,,,10.5,0.0,0.01,0.1,0.97,,,1,2026-04-10 11:07:16.0,,,,,0,,,51,,,10.6,0.0,0.01,0.11,0.98,,,,,,,0,,,42,,,10.3,0.0,0.01,0.1,0.96,,,SDOT001,residential_area,2026-04-10_11:07:00,V02Q1940526


In [0]:
from pyspark.sql.functions import col, coalesce, lit

target_districts = ["Gangnam-gu", "Songpa-gu", "Gangdong-gu"]

df_silver = df_flat.filter(
    col("AUTONOMOUS_DISTRICT").isin(target_districts)
).select(
    col("AUTONOMOUS_DISTRICT").alias("district"),
    col("ADMINISTRATIVE_DISTRICT").alias("admin_district"),
    col("REGION").alias("region_type"),
    coalesce(col("MAX_NOISE").cast("double"), lit(0.0)).alias("max_noise"),
    coalesce(col("AVG_NOISE").cast("double"), lit(0.0)).alias("avg_noise"),
    coalesce(col("AVG_VIBR_X").cast("double"), lit(0.0)).alias("avg_vibr_x"),
    coalesce(col("AVG_VIBR_Y").cast("double"), lit(0.0)).alias("avg_vibr_y"),
    coalesce(col("AVG_VIBR_Z").cast("double"), lit(0.0)).alias("avg_vibr_z"),
    col("DATE").alias("collected_at"),
    col("DATA_NO").alias("data_no")
)

display(df_silver)

district,admin_district,region_type,max_noise,avg_noise,avg_vibr_x,avg_vibr_y,avg_vibr_z,collected_at,data_no
Gangdong-gu,Cheonho2(i)-dong,residential_area,67.0,64.0,0.09,0.02,0.94,2026-04-10 11:07:57.0,1
Gangdong-gu,Cheonho2(i)-dong,residential_area,68.0,65.0,0.02,0.02,1.06,2026-04-10 11:07:57.0,1
Gangdong-gu,Cheonho2(i)-dong,residential_area,67.0,64.0,0.01,0.04,1.04,2026-04-10 11:07:57.0,1
Gangdong-gu,Seongnae3(sam)-dong,residential_area,71.0,67.0,0.08,0.02,1.01,2026-04-10 11:07:57.0,1
Gangdong-gu,Cheonho1(il)-dong,residential_area,70.0,65.0,0.01,0.08,1.01,2026-04-10 11:07:57.0,1
Gangdong-gu,Seongnae3(sam)-dong,residential_area,68.0,65.0,0.01,0.02,1.03,2026-04-10 11:07:57.0,1
Gangdong-gu,Myeon-gil1(il)-dong,residential_area,67.0,64.0,0.05,0.06,1.04,2026-04-10 11:07:56.0,1
Gangdong-gu,Godeok1(il)-dong,roads_and_parks,75.0,73.0,0.0,0.0,0.0,2026-04-10 11:07:54.0,1
Gangdong-gu,Amsa3(sam)-dong,parks,61.0,58.0,0.0,0.0,0.0,2026-04-10 11:07:54.0,1
Gangdong-gu,Cheonho2(i)-dong,residential_area,62.0,61.0,0.0,0.0,0.0,2026-04-10 11:07:54.0,1


In [0]:
display(df_silver.groupBy("data_no").count())

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8148179673071526>, line 4
      1 from pyspark.sql.functions import col, coalesce, lit
      3 df_silver = df_flat.filter(
----> 4     col("AUTONOMOUS_DISTRICT").isin(target_districts)
      5 ).select(
      6     col("AUTONOMOUS_DISTRICT").alias("district"),
      7     col("ADMINISTRATIVE_DISTRICT").alias("admin_district"),
      8     col("REGION").alias("region_type"),
      9     coalesce(col("MAX_NOISE").cast("double"), lit(0.0)).alias("max_noise"),
     10     coalesce(col("AVG_NOISE").cast("double"), lit(0.0)).alias("avg_noise"),
     11     coalesce(col("AVG_VIBR_X").cast("double"), lit(0.0)).alias("avg_vibr_x"),
     12     coalesce(col("AVG_VIBR_Y").cast("double"), lit(0.0)).alias("avg_vibr_y"),
     13     coalesce(col("AVG_VIBR_Z").cast("double"), lit(0.0)).alias("avg_vibr_z"),
     14     col("DATE").alias("

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.sdot")

In [0]:
display(spark.sql("SELECT * FROM silver.sdot LIMIT 5"))

district,admin_district,region_type,max_noise,avg_noise,avg_vibr_x,avg_vibr_y,avg_vibr_z,collected_at,data_no
Gangdong-gu,Cheonho2(i)-dong,residential_area,67.0,64.0,0.09,0.02,0.94,2026-04-10 11:07:57.0,1
Gangdong-gu,Cheonho2(i)-dong,residential_area,68.0,65.0,0.02,0.02,1.06,2026-04-10 11:07:57.0,1
Gangdong-gu,Cheonho2(i)-dong,residential_area,67.0,64.0,0.01,0.04,1.04,2026-04-10 11:07:57.0,1
Gangdong-gu,Seongnae3(sam)-dong,residential_area,71.0,67.0,0.08,0.02,1.01,2026-04-10 11:07:57.0,1
Gangdong-gu,Cheonho1(il)-dong,residential_area,70.0,65.0,0.01,0.08,1.01,2026-04-10 11:07:57.0,1
